# College ROI Analysis: Data Cleaning and Exploratory Analysis

**Project goal:** Analyze U.S. college bachelor's degree programs to compare student debt, 5-year earnings, earnings growth, and debt-to-earnings return on investment (ROI).

This notebook documents the cleaning and early analysis process using College Scorecard field-of-study data.

## Import libraries

We use `pandas` for loading, cleaning, grouping, and summarizing tabular data. `numpy` is useful for missing values and numerical work.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

## Load raw field-of-study data

The raw data is stored locally in `data/raw/college_raw_data/`. Because the raw files are large, they should be excluded from GitHub using `.gitignore`.

In [2]:
# Path to the raw College Scorecard field-of-study file.
# The ../ means "go up one folder" because this notebook lives inside the notebooks/ folder.
raw_file = Path("../data/raw/college_raw_data/Most-Recent-Cohorts-Field-of-Study.csv")

# low_memory=False prevents pandas from guessing column data types in chunks.
df = pd.read_csv(raw_file, low_memory=False)

# Shape shows (rows, columns).
df.shape

(227980, 178)

## Initial data inspection

Before cleaning, we inspect the dataset structure: sample rows, data types, column names, and key value counts. This helps confirm what each row represents and whether the columns needed for analysis are available.

In [3]:
# Preview the first few rows.
df.head()

,UNITID,OPEID6,INSTNM,CONTROL,MAIN,CIPCODE,CIPDESC,CREDLEV,CREDDESC,IPEDSCOUNT1,...,EARN_COUNT_MALE_WNE_5YR,EARN_MALE_WNE_MDN_5YR,EARN_COUNT_NOMALE_WNE_5YR,EARN_NOMALE_WNE_MDN_5YR,EARN_COUNT_HIGH_CRED_5YR,EARN_IN_STATE_5YR,EARN_COUNT_WNE_4YR_NAT,EARN_MDN_4YR_NAT,EARN_P25_4YR_NAT,EARN_P75_4YR_NAT
0,100654.0,1002,Alabama A & M University,Public,1,109,Animal Sciences.,3,Bachelor's Degree,9.0,...,PS,PS,PS,PS,PS,PS,6104,49634,36349,66880
1,100654.0,1002,Alabama A & M University,Public,1,110,Food Science and Technology.,3,Bachelor's Degree,7.0,...,PS,PS,PS,PS,PS,PS,1301,70873,54113,88238
2,100654.0,1002,Alabama A & M University,Public,1,110,Food Science and Technology.,5,Master's Degree,7.0,...,PS,PS,PS,PS,PS,PS,380,88332,65085,115796
3,100654.0,1002,Alabama A & M University,Public,1,110,Food Science and Technology.,6,Doctoral Degree,3.0,...,PS,PS,PS,PS,PS,PS,PS,PS,PS,PS
4,100654.0,1002,Alabama A & M University,Public,1,199,Agricultural/Animal/Plant/Veterinary Science a...,3,Bachelor's Degree,1.0,...,PS,PS,PS,PS,PS,PS,413,65543,46023,83774


In [4]:
# Review column types, non-null counts, and memory usage.
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 227980 entries, 0 to 227979
Columns: 178 entries, UNITID to EARN_P75_4YR_NAT
dtypes: float64(3), int64(5), object(170)
memory usage: 309.6+ MB


In [5]:
# Display all column names so we can identify debt and earnings variables.
df.columns.tolist()

['UNITID',
 'OPEID6',
 'INSTNM',
 'CONTROL',
 'MAIN',
 'CIPCODE',
 'CIPDESC',
 'CREDLEV',
 'CREDDESC',
 'IPEDSCOUNT1',
 'IPEDSCOUNT2',
 'DEBT_ALL_STGP_ANY_N',
 'DEBT_ALL_STGP_ANY_MEAN',
 'DEBT_ALL_STGP_ANY_MDN',
 'DEBT_ALL_STGP_EVAL_N',
 'DEBT_ALL_STGP_EVAL_MEAN',
 'DEBT_ALL_STGP_EVAL_MDN',
 'DEBT_ALL_PP_ANY_N',
 'DEBT_ALL_PP_ANY_MEAN',
 'DEBT_ALL_PP_ANY_MDN',
 'DEBT_ALL_PP_EVAL_N',
 'DEBT_ALL_PP_EVAL_MEAN',
 'DEBT_ALL_PP_EVAL_MDN',
 'DEBT_MALE_STGP_ANY_N',
 'DEBT_MALE_STGP_ANY_MEAN',
 'DEBT_MALE_STGP_ANY_MDN',
 'DEBT_MALE_STGP_EVAL_N',
 'DEBT_MALE_STGP_EVAL_MEAN',
 'DEBT_MALE_STGP_EVAL_MDN',
 'DEBT_MALE_PP_ANY_N',
 'DEBT_MALE_PP_ANY_MEAN',
 'DEBT_MALE_PP_ANY_MDN',
 'DEBT_MALE_PP_EVAL_N',
 'DEBT_MALE_PP_EVAL_MEAN',
 'DEBT_MALE_PP_EVAL_MDN',
 'DEBT_NOTMALE_STGP_ANY_N',
 'DEBT_NOTMALE_STGP_ANY_MEAN',
 'DEBT_NOTMALE_STGP_ANY_MDN',
 'DEBT_NOTMALE_STGP_EVAL_N',
 'DEBT_NOTMALE_STGP_EVAL_MEAN',
 'DEBT_NOTMALE_STGP_EVAL_MDN',
 'DEBT_NOTMALE_PP_ANY_N',
 'DEBT_NOTMALE_PP_ANY_MEAN',
 'DEBT_NOTMAL

## Select relevant columns

For this project, we focus on:

- `INSTNM`: Institution name
- `CIPDESC`: Field of study / major
- `CREDDESC`: Credential level
- `DEBT_ALL_STGP_ANY_MDN`: Median debt
- `EARN_MDN_1YR`: Median earnings 1 year after completion
- `EARN_MDN_4YR`: Median earnings 4 years after completion
- `EARN_MDN_5YR`: Median earnings 5 years after completion

In [6]:
key_columns = [
    'INSTNM',
    'CIPDESC',
    'CREDDESC',
    'DEBT_ALL_STGP_ANY_MDN',
    'EARN_MDN_1YR',
    'EARN_MDN_4YR',
    'EARN_MDN_5YR'
]

# Preview the fields we plan to use.
df[key_columns].head(20)

,INSTNM,CIPDESC,CREDDESC,DEBT_ALL_STGP_ANY_MDN,EARN_MDN_1YR,EARN_MDN_4YR,EARN_MDN_5YR
0,Alabama A & M University,Animal Sciences.,Bachelor's Degree,PS,PS,PS,PS
1,Alabama A & M University,Food Science and Technology.,Bachelor's Degree,PS,PS,PS,PS
2,Alabama A & M University,Food Science and Technology.,Master's Degree,PS,PS,PS,PS
3,Alabama A & M University,Food Science and Technology.,Doctoral Degree,PS,PS,PS,PS
4,Alabama A & M University,Agricultural/Animal/Plant/Veterinary Science a...,Bachelor's Degree,PS,PS,PS,PS
5,Alabama A & M University,Agricultural/Animal/Plant/Veterinary Science a...,Master's Degree,PS,PS,PS,PS
6,Alabama A & M University,Agricultural/Animal/Plant/Veterinary Science a...,Doctoral Degree,PS,PS,PS,PS
7,Alabama A & M University,Forestry.,Bachelor's Degree,PS,PS,64749,PS
8,Alabama A & M University,"City/Urban, Community, and Regional Planning.",Bachelor's Degree,PS,PS,PS,PS
9,Alabama A & M University,"City/Urban, Community, and Regional Planning.",Master's Degree,PS,PS,PS,PS


## Check privacy-suppressed values

The dataset uses `PS` for privacy-suppressed values. These are not usable as numbers, so we need to measure how common they are before analysis.

In [7]:
# Count privacy-suppressed values in our key numeric columns.
key_numeric_columns = [
    'DEBT_ALL_STGP_ANY_MDN',
    'EARN_MDN_1YR',
    'EARN_MDN_4YR',
    'EARN_MDN_5YR'
]

df[key_numeric_columns].eq('PS').sum()

DEBT_ALL_STGP_ANY_MDN    164711
EARN_MDN_1YR             156386
EARN_MDN_4YR             169878
EARN_MDN_5YR             158102
dtype: int64

## Filter to bachelor's degree programs

To keep the project focused and easier to interpret, this first analysis only uses bachelor's degree programs. Other credential levels can be explored in a future expansion.

In [8]:
# Check how many rows exist for each credential level.
df['CREDDESC'].value_counts()

CREDDESC
Bachelor's Degree                                                        71664
Undergraduate Certificate or Diploma                                     43465
Associate's Degree                                                       42373
Master's Degree                                                          38621
Graduate/Professional Certificate                                        13921
Doctoral Degree                                                          12637
First Professional Degree                                                 2350
Non-Credential Program (Preparatory Coursework/Teacher Certification)     1700
Post-baccalaureate Certificate                                            1249
Name: count, dtype: int64

In [9]:
# Keep only bachelor's degree rows.
bachelors = df[df['CREDDESC'] == "Bachelor's Degree"].copy()

bachelors.shape

(71664, 178)

## Build the analysis dataset

We keep bachelor's degree records with available debt and 5-year earnings values. This creates the main project dataset for debt, earnings, and ROI analysis.

In [10]:
# Keep rows where debt and 5-year earnings are not privacy suppressed.
# We will still convert non-numeric values to NaN later.
roi_df = bachelors[
    (bachelors['DEBT_ALL_STGP_ANY_MDN'] != 'PS') &
    (bachelors['EARN_MDN_5YR'] != 'PS')
][key_columns].copy()

roi_df.shape

(24265, 7)

## Convert debt and earnings columns to numeric values

These columns loaded as `object` because they originally contained a mix of numbers and text values such as `PS` or missing codes. `pd.to_numeric(..., errors='coerce')` converts valid numbers and changes invalid values to `NaN`.

In [11]:
# Check data types before conversion.
roi_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 24265 entries, 12 to 227961
Data columns (total 7 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   INSTNM                 24265 non-null  object
 1   CIPDESC                24265 non-null  object
 2   CREDDESC               24265 non-null  object
 3   DEBT_ALL_STGP_ANY_MDN  21092 non-null  object
 4   EARN_MDN_1YR           21167 non-null  object
 5   EARN_MDN_4YR           24265 non-null  object
 6   EARN_MDN_5YR           21167 non-null  object
dtypes: object(7)
memory usage: 1.5+ MB


In [12]:
# Convert debt and earnings columns from text/object into numeric columns.
for col in key_numeric_columns:
    roi_df[col] = pd.to_numeric(roi_df[col], errors='coerce')

# Verify that numeric columns are now float64.
roi_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 24265 entries, 12 to 227961
Data columns (total 7 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   INSTNM                 24265 non-null  object 
 1   CIPDESC                24265 non-null  object 
 2   CREDDESC               24265 non-null  object 
 3   DEBT_ALL_STGP_ANY_MDN  21092 non-null  float64
 4   EARN_MDN_1YR           19877 non-null  float64
 5   EARN_MDN_4YR           20403 non-null  float64
 6   EARN_MDN_5YR           21167 non-null  float64
dtypes: float64(4), object(3)
memory usage: 1.5+ MB


In [13]:
# Summary statistics for the two most important numeric columns.
roi_df[['DEBT_ALL_STGP_ANY_MDN', 'EARN_MDN_5YR']].describe()

,DEBT_ALL_STGP_ANY_MDN,EARN_MDN_5YR
count,21092.000000,21167.000000
mean,24883.834297,63530.321727
std,6869.894500,21130.176006
min,2750.000000,14612.000000
25%,21500.000000,48779.500000
50%,25000.000000,58919.000000
75%,27000.000000,74585.000000
max,57500.000000,298016.000000


## Create calculated metrics

We create three new metrics:

- `ROI_RATIO`: 5-year earnings divided by median debt
- `EARN_GROWTH`: dollar growth from year 1 to year 5
- `GROWTH_RATIO`: 5-year earnings divided by year-1 earnings

`EARN_GROWTH` shows absolute dollar growth, while `GROWTH_RATIO` shows relative growth.

In [14]:
# Debt-to-earnings ROI ratio.
roi_df['ROI_RATIO'] = roi_df['EARN_MDN_5YR'] / roi_df['DEBT_ALL_STGP_ANY_MDN']

# Absolute earnings growth in dollars.
roi_df['EARN_GROWTH'] = roi_df['EARN_MDN_5YR'] - roi_df['EARN_MDN_1YR']

# Relative earnings growth ratio.
roi_df['GROWTH_RATIO'] = roi_df['EARN_MDN_5YR'] / roi_df['EARN_MDN_1YR']

roi_df[[
    'CIPDESC',
    'DEBT_ALL_STGP_ANY_MDN',
    'EARN_MDN_1YR',
    'EARN_MDN_5YR',
    'ROI_RATIO',
    'EARN_GROWTH',
    'GROWTH_RATIO'
]].head()

,CIPDESC,DEBT_ALL_STGP_ANY_MDN,EARN_MDN_1YR,EARN_MDN_5YR,ROI_RATIO,EARN_GROWTH,GROWTH_RATIO
12,"Computer and Information Sciences, General.",31000.0,63900.0,85218.0,2.748968,21318.0,1.333615
24,"Electrical, Electronics, and Communications En...",35000.0,72241.0,90409.0,2.583114,18168.0,1.251492
26,Mechanical Engineering.,30500.0,71954.0,82929.0,2.718984,10975.0,1.152528
32,"Liberal Arts and Sciences, General Studies and...",33000.0,28877.0,46627.0,1.412939,17750.0,1.614676
33,"Biology, General.",28271.0,29098.0,40721.0,1.440381,11623.0,1.399443


## Inspect ROI distribution and outliers

ROI can be heavily influenced by very low debt values. We inspect the distribution and highest ROI records before using the metric in summaries.

In [15]:
# Summary statistics for the ROI ratio.
roi_df['ROI_RATIO'].describe()

count    21092.000000
mean         2.780419
std          1.436371
min          0.417793
25%          1.902323
50%          2.470011
75%          3.299383
max         37.226545
Name: ROI_RATIO, dtype: float64

In [16]:
# View the highest individual ROI records to understand possible outliers.
roi_df.sort_values(
    by='ROI_RATIO',
    ascending=False
)[[
    'INSTNM',
    'CIPDESC',
    'DEBT_ALL_STGP_ANY_MDN',
    'EARN_MDN_5YR',
    'ROI_RATIO'
]].head(20)

,INSTNM,CIPDESC,DEBT_ALL_STGP_ANY_MDN,EARN_MDN_5YR,ROI_RATIO
26706,University of Southern California,"Engineering, General.",2750.0,102373.0,37.226545
79223,Harvard University,Economics.,6617.0,161251.0,24.369201
199329,Stanford University,Computer Science.,10399.0,247797.0,23.828926
26788,University of Southern California,"Biochemistry, Biophysics and Molecular Biology.",3750.0,88746.0,23.665600
158677,Brown University,Computer Science.,11500.0,271601.0,23.617478
80399,Massachusetts Institute of Technology,Mathematics.,10003.0,226193.0,22.612516
33428,Yale University,"Computer and Information Sciences, General.",12750.0,271466.0,21.291451
154740,University of Pennsylvania,"Computer and Information Sciences, General.",15000.0,298016.0,19.867733
26781,University of Southern California,"Liberal Arts and Sciences, General Studies and...",3250.0,64040.0,19.704615
155025,University of Pennsylvania,Finance and Financial Management Services.,12865.0,242357.0,18.838476


## Create major-level summary table

Because individual institution-major records can be noisy, we aggregate by major. We include both mean and median metrics so we can compare whether outliers are influencing the averages.

In [17]:
major_summary = roi_df.groupby('CIPDESC').agg(
    mean_roi=('ROI_RATIO', 'mean'),
    median_roi=('ROI_RATIO', 'median'),
    mean_debt=('DEBT_ALL_STGP_ANY_MDN', 'mean'),
    median_debt=('DEBT_ALL_STGP_ANY_MDN', 'median'),
    mean_earnings=('EARN_MDN_5YR', 'mean'),
    median_earnings=('EARN_MDN_5YR', 'median'),
    mean_growth=('EARN_GROWTH', 'mean'),
    median_growth=('EARN_GROWTH', 'median'),
    mean_growth_ratio=('GROWTH_RATIO', 'mean'),
    median_growth_ratio=('GROWTH_RATIO', 'median'),
    count=('ROI_RATIO', 'count')
).dropna()

major_summary.head()

,mean_roi,median_roi,mean_debt,median_debt,mean_earnings,median_earnings,mean_growth,median_growth,mean_growth_ratio,median_growth_ratio,count
CIPDESC,,,,,,,,,,,
Accounting and Related Services.,3.118566,3.018446,26078.703364,25000.0,73048.385084,71752.0,18640.022329,18443.0,1.352131,1.340175,654
"Aerospace, Aeronautical, and Astronautical/Space Engineering.",4.258939,3.916917,23693.800000,24846.0,97584.127273,97915.0,23713.943396,23567.0,1.328191,1.330020,55
Agricultural Business and Management.,3.508456,3.171243,19846.419355,20000.0,66459.516129,63223.5,17590.295082,17431.0,1.363682,1.368438,62
Agricultural Engineering.,3.912696,3.518523,22485.562500,22750.0,84530.625000,83346.0,18900.437500,16568.0,1.305295,1.240082,16
Agricultural Mechanization.,3.935630,3.423925,19199.166667,20608.0,70581.333333,72288.0,19011.333333,18639.0,1.448365,1.338327,6


## Apply minimum sample-size filter

Some majors only have a few usable observations. To make rankings more reliable, we only include majors with at least 20 observations.

In [18]:
# Understand how many observations each major has.
major_summary['count'].describe()

count     283.000000
mean       74.522968
std       152.393488
min         1.000000
25%         5.000000
50%        19.000000
75%        59.000000
max      1086.000000
Name: count, dtype: float64

In [19]:
# Keep majors with at least 20 observations.
MIN_OBSERVATIONS = 20

major_summary_filtered = major_summary[
    major_summary['count'] >= MIN_OBSERVATIONS
].copy()

major_summary_filtered.shape

(138, 11)

## Research question 1: Which majors have the highest debt?

This ranking identifies majors with the highest average median debt among majors with at least 20 observations.

In [20]:
highest_debt = major_summary_filtered.sort_values(
    by='mean_debt',
    ascending=False
)

highest_debt.head(20)

,mean_roi,median_roi,mean_debt,median_debt,mean_earnings,median_earnings,mean_growth,median_growth,mean_growth_ratio,median_growth_ratio,count
CIPDESC,,,,,,,,,,,
Computer Systems Analysis.,2.166607,1.621413,39817.500000,46000.0,76928.681818,74585.0,23961.238095,22780.0,1.447880,1.439726,22
Electrical/Electronic Engineering Technologies/Technicians.,2.813954,2.607593,36539.060606,31000.0,85554.848485,85005.0,18588.718750,17610.0,1.283611,1.261295,33
Health and Medical Administrative Services.,2.006235,1.900975,35012.073394,31000.0,59042.788991,56839.0,13790.125000,12109.0,1.316004,1.306246,218
Computer Software and Media Applications.,2.114588,1.526741,34446.716981,36916.0,58414.169811,53738.0,18586.288462,15810.5,1.504381,1.462546,53
Computer Systems Networking and Telecommunications.,2.654528,2.235330,33953.815789,27000.0,76772.631579,73315.0,16855.157895,12775.0,1.280543,1.211018,38
"Human Services, General.",1.566656,1.459023,33275.544118,30504.5,45465.235294,44584.5,7724.737705,7385.0,1.216073,1.182555,68
Computer Programming.,2.776043,2.669694,32636.291667,31050.0,84622.541667,82894.0,24519.416667,30344.0,1.456747,1.539901,24
Computer/Information Technology Administration and Management.,2.890535,2.849654,32604.183333,29484.5,83669.983333,77609.0,22731.172414,22109.5,1.374451,1.333380,60
Audiovisual Communications Technologies/Technicians.,1.535056,1.443682,31785.850000,27000.0,41204.227273,38309.0,16812.000000,17352.0,1.869628,1.771400,20


## Research question 2: Which majors have the highest earnings?

This ranking identifies majors with the highest average median earnings five years after completion.

In [21]:
highest_earnings = major_summary_filtered.sort_values(
    by='mean_earnings',
    ascending=False
)

highest_earnings.head(20)

,mean_roi,median_roi,mean_debt,median_debt,mean_earnings,median_earnings,mean_growth,median_growth,mean_growth_ratio,median_growth_ratio,count
CIPDESC,,,,,,,,,,,
Computer Engineering.,5.359038,4.656175,23038.672131,23668.5,114191.073770,108654.0,31707.325000,28007.5,1.396829,1.341341,122
Computer Science.,5.477572,4.445500,22885.695473,23026.0,111296.255144,106149.0,36415.760504,32854.5,1.503987,1.457103,243
"Pharmacy, Pharmaceutical Sciences, and Administration.",4.997507,5.169815,23135.600000,25250.0,107687.476190,100306.0,39312.363636,38246.0,1.983035,1.732730,20
"Electrical, Electronics, and Communications Engineering.",4.486357,4.021247,23787.986667,25000.0,100944.084444,99007.0,23180.121622,21523.0,1.302924,1.270551,225
Chemical Engineering.,4.451611,3.970823,23173.680556,23274.0,98121.520833,97009.5,24794.958042,23317.0,1.351100,1.314083,144
"Aerospace, Aeronautical, and Astronautical/Space Engineering.",4.258939,3.916917,23693.800000,24846.0,97584.127273,97915.0,23713.943396,23567.0,1.328191,1.330020,55
Industrial Engineering.,4.159731,3.835463,23987.072464,24989.0,95855.072464,93671.0,22600.164179,19922.0,1.309285,1.269776,69
Biomedical/Medical Engineering.,4.452604,4.119043,22581.522727,22900.5,95067.079545,94444.0,29600.220930,28468.5,1.476510,1.429345,88
Applied Mathematics.,5.492469,4.639202,18987.363636,17678.0,94707.454545,90036.0,35402.409091,34678.0,1.648218,1.552994,22


## Research question 3: Which majors show the strongest earnings growth?

We look at both absolute dollar growth and relative growth from year 1 to year 5.

In [22]:
highest_growth_amount = major_summary_filtered.sort_values(
    by='mean_growth',
    ascending=False
)

highest_growth_amount.head(20)

,mean_roi,median_roi,mean_debt,median_debt,mean_earnings,median_earnings,mean_growth,median_growth,mean_growth_ratio,median_growth_ratio,count
CIPDESC,,,,,,,,,,,
"Pharmacy, Pharmaceutical Sciences, and Administration.",4.997507,5.169815,23135.600000,25250.0,107687.476190,100306.0,39312.363636,38246.0,1.983035,1.732730,20
Neurobiology and Neurosciences.,3.403919,3.097928,21334.080645,22093.0,67954.596774,66952.5,36756.719298,37434.0,2.252858,2.286283,62
Computer Science.,5.477572,4.445500,22885.695473,23026.0,111296.255144,106149.0,36415.760504,32854.5,1.503987,1.457103,243
Cell/Cellular Biology and Anatomical Sciences.,3.857087,3.522680,19839.062500,19437.5,72421.343750,70612.5,36222.900000,36005.0,2.080007,1.990251,32
Physics.,3.766449,3.429011,22389.127660,22750.0,81739.234043,80500.0,36075.297297,35460.0,1.858590,1.852837,47
Applied Mathematics.,5.492469,4.639202,18987.363636,17678.0,94707.454545,90036.0,35402.409091,34678.0,1.648218,1.552994,22
"Physiology, Pathology and Related Sciences.",2.863785,2.732537,23062.166667,24250.0,63498.055556,66735.5,34364.208333,34556.0,2.159756,2.132068,54
Communication Disorders Sciences and Services.,2.805058,2.651300,21823.956989,21927.0,58781.725806,58669.5,33279.688889,32897.0,2.443643,2.273724,186
Statistics.,5.063388,4.537376,19538.620690,20150.0,93623.931034,92382.0,32546.642857,30443.5,1.548392,1.514879,29


In [23]:
highest_growth_ratio = major_summary_filtered.sort_values(
    by='mean_growth_ratio',
    ascending=False
)

highest_growth_ratio.head(20)

,mean_roi,median_roi,mean_debt,median_debt,mean_earnings,median_earnings,mean_growth,median_growth,mean_growth_ratio,median_growth_ratio,count
CIPDESC,,,,,,,,,,,
Communication Disorders Sciences and Services.,2.805058,2.651300,21823.956989,21927.0,58781.725806,58669.5,33279.688889,32897.0,2.443643,2.273724,186
Neurobiology and Neurosciences.,3.403919,3.097928,21334.080645,22093.0,67954.596774,66952.5,36756.719298,37434.0,2.252858,2.286283,62
"Physiology, Pathology and Related Sciences.",2.863785,2.732537,23062.166667,24250.0,63498.055556,66735.5,34364.208333,34556.0,2.159756,2.132068,54
Cell/Cellular Biology and Anatomical Sciences.,3.857087,3.522680,19839.062500,19437.5,72421.343750,70612.5,36222.900000,36005.0,2.080007,1.990251,32
Health/Medical Preparatory Programs.,2.657977,2.657562,24777.937500,25000.0,63547.942857,60119.0,31217.931034,29713.0,2.024117,1.901775,32
"Pharmacy, Pharmaceutical Sciences, and Administration.",4.997507,5.169815,23135.600000,25250.0,107687.476190,100306.0,39312.363636,38246.0,1.983035,1.732730,20
"Biology, General.",2.688813,2.523549,23789.185504,25000.0,60819.075887,61102.0,28971.269337,28776.0,1.968717,1.868465,814
Nutrition Sciences.,2.579127,2.303742,23296.696970,24710.0,57218.939394,54740.0,26897.366667,27155.0,1.939338,1.910009,33
"Linguistic, Comparative, and Related Language Studies and Services.",2.647444,2.475077,20497.824561,19912.0,50212.052632,49113.0,22971.176471,21706.0,1.927972,1.921357,57


In [27]:
roi_df['EARN_GROWTH'] = (
    roi_df['EARN_MDN_5YR']
    - roi_df['EARN_MDN_1YR']
)

roi_df['GROWTH_RATIO'] = (
    roi_df['EARN_MDN_5YR']
    / roi_df['EARN_MDN_1YR']
)

growth_summary = roi_df.groupby('CIPDESC').agg(
    avg_growth=('EARN_GROWTH', 'mean'),
    avg_growth_ratio=('GROWTH_RATIO', 'mean'),
    count=('ROI_RATIO', 'count')
)

growth_summary = growth_summary[
    growth_summary['count'] >= 20
]

growth_summary.sort_values(
    by='avg_growth',
    ascending=False
).head(20)

,avg_growth,avg_growth_ratio,count
CIPDESC,,,
"Pharmacy, Pharmaceutical Sciences, and Administration.",39312.363636,1.983035,20
Neurobiology and Neurosciences.,36756.719298,2.252858,62
Computer Science.,36415.760504,1.503987,243
Cell/Cellular Biology and Anatomical Sciences.,36222.900000,2.080007,32
Physics.,36075.297297,1.858590,47
Applied Mathematics.,35402.409091,1.648218,22
"Physiology, Pathology and Related Sciences.",34364.208333,2.159756,54
Communication Disorders Sciences and Services.,33279.688889,2.443643,186
Statistics.,32546.642857,1.548392,29


## Research question 4: Which majors have the strongest debt-to-earnings ROI?

This ranking compares 5-year earnings against median debt. We use the sample-size filter to reduce unstable rankings caused by majors with very few observations.

In [24]:
highest_roi = major_summary_filtered.sort_values(
    by='mean_roi',
    ascending=False
)

highest_roi.head(20)

,mean_roi,median_roi,mean_debt,median_debt,mean_earnings,median_earnings,mean_growth,median_growth,mean_growth_ratio,median_growth_ratio,count
CIPDESC,,,,,,,,,,,
Applied Mathematics.,5.492469,4.639202,18987.363636,17678.0,94707.454545,90036.0,35402.409091,34678.0,1.648218,1.552994,22
Computer Science.,5.477572,4.445500,22885.695473,23026.0,111296.255144,106149.0,36415.760504,32854.5,1.503987,1.457103,243
Computer Engineering.,5.359038,4.656175,23038.672131,23668.5,114191.073770,108654.0,31707.325000,28007.5,1.396829,1.341341,122
Statistics.,5.063388,4.537376,19538.620690,20150.0,93623.931034,92382.0,32546.642857,30443.5,1.548392,1.514879,29
"Pharmacy, Pharmaceutical Sciences, and Administration.",4.997507,5.169815,23135.600000,25250.0,107687.476190,100306.0,39312.363636,38246.0,1.983035,1.732730,20
"Engineering, General.",4.782743,3.274648,24459.000000,27000.0,91259.555556,88354.5,21463.771429,19986.0,1.317078,1.290296,36
"Electrical, Electronics, and Communications Engineering.",4.486357,4.021247,23787.986667,25000.0,100944.084444,99007.0,23180.121622,21523.0,1.302924,1.270551,225
Biomedical/Medical Engineering.,4.452604,4.119043,22581.522727,22900.5,95067.079545,94444.0,29600.220930,28468.5,1.476510,1.429345,88
Chemical Engineering.,4.451611,3.970823,23173.680556,23274.0,98121.520833,97009.5,24794.958042,23317.0,1.351100,1.314083,144


## Preliminary findings

These findings are preliminary and should be refined after visualization in Power BI.

- Applied Mathematics, Computer Science, Computer Engineering, and Statistics ranked among the highest ROI majors after filtering to majors with at least 20 observations.
- Engineering-related disciplines frequently appeared among the highest earning and highest ROI majors.
- Some extreme ROI values were driven by very low debt values, so individual records were inspected and rankings were filtered by sample size.
- Mean and median metrics are both included to help evaluate the influence of outliers.

## Save cleaned datasets

The raw files are too large for GitHub, so this project saves smaller cleaned files that can be uploaded and used in Power BI. The raw data should remain local and be excluded with `.gitignore`.

In [25]:
# Create cleaned data folder if it does not already exist.
cleaned_dir = Path("../data/cleaned")
cleaned_dir.mkdir(parents=True, exist_ok=True)

# Save detailed cleaned data.
roi_df.to_csv(cleaned_dir / "roi_df.csv", index=False)

# Save major-level summary data for Power BI.
major_summary_filtered.to_csv(cleaned_dir / "major_summary.csv", index=True)

print("Cleaned files saved successfully.")

Cleaned files saved successfully.


## Next steps

1. Build a Power BI dashboard using `major_summary.csv`.
2. Create visuals for debt, earnings, ROI, and earnings growth.
3. Write final findings after reviewing the dashboard visuals.